# Diplomatic Structural Ties Development

This notebook develops and validates the diplomatic structural-ties component of the US-China Closeness Index.

The score answers:

> How institutionally deep is a country's diplomatic representation relationship with China or the United States in a given year?

It does not measure whether the relationship is currently healthy. Diplomatic structural depth and relationship condition remain separate analytical layers.

## Established source definitions

The source provides directed diplomatic relations. Each country-anchor-year can contain:

- representation maintained by the anchor in the country; and
- representation maintained by the country in the anchor.

`LOR`, or Level of Representation, is the source's zero-to-one index combining formal accreditation with mission focus.

| LOR | Interpretation |
|---:|---|
| 1.000 | Ambassador-level, single focus |
| 0.750 | Unknown, charge d'affaires, or minister, single focus |
| 0.500 | Ambassador-level, multiple focus |
| 0.375 | Unknown, charge d'affaires, or minister, multiple focus |
| 0.125 | Interest desk |
| 0.100 | Interests served by |
| 0.000 | Expelled, recalled, withdrawn, or severed |

A missing directed record is scored as zero operational depth but remains explicitly identified as unobserved. An observed `LOR = 0` remains identifiable as an adverse diplomatic observation.

## Baseline score

Let:

```text
a = anchor-to-country LOR
b = country-to-anchor LOR
```

Calculate:

```text
mean_depth = (a + b) / 2
reciprocity_depth = min(a, b)
```

The baseline is:

```text
diplomatic_structural_score_raw
    = 0.75 × mean_depth
    + 0.25 × reciprocity_depth
```

This is equivalent to weighting the stronger direction by 37.5% and the weaker direction by 62.5%.

The baseline treats overall depth as primary while giving additional weight to representation sustained in both directions. Mean-only, equal-component-weight, and geometric alternatives are retained for sensitivity analysis.

## Continuity policy

Continuity is calculated as a diagnostic but does not enter the baseline score.

This avoids rewarding countries merely because they have existed or been observed for longer. The production panel retains recent presence rates and current presence streaks for later robustness work.

In [1]:
# locate repository and define paths
from pathlib import Path
import subprocess
import sys

import pandas as pd


def locate_repository_root(start: Path | None = None) -> Path:
    try:
        start_path = (start or Path.cwd()).resolve()
    except FileNotFoundError:
        start_path = Path.home()

    candidates = [start_path]
    candidates.extend(start_path.parents)

    for candidate in candidates:
        if (
            (candidate / "src" / "data").exists()
            and (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = locate_repository_root()
SCRIPT = REPOSITORY_ROOT / "src" / "data" / "build_diplomatic_structural_ties.py"
DIRECTED_PANEL = (
    REPOSITORY_ROOT
    / "data"
    / "interim"
    / "relations"
    / "drd_anchor_directed_country_year.parquet"
)
PRODUCTION_PANEL = (
    REPOSITORY_ROOT
    / "data"
    / "processed"
    / "diplomatic_structural_ties_country_anchor_year.csv"
)
REPORT_DIRECTORY = (
    REPOSITORY_ROOT
    / "outputs"
    / "tables"
    / "diplomatic_structural_ties"
)

print("Repository root located.")
print("Builder exists:", SCRIPT.exists())
print("Directed DRD panel exists:", DIRECTED_PANEL.exists())

Repository root located.
Builder exists: True
Directed DRD panel exists: True


In [2]:
# run the diplomatic structural-ties builder
completed = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=REPOSITORY_ROOT,
    check=False,
    text=True,
    capture_output=True,
)

if completed.stdout.strip():
    print(completed.stdout)

if completed.stderr.strip():
    print("SCRIPT STDERR:")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        f"{SCRIPT.name} failed with exit code {completed.returncode}."
    )

DIPLOMATIC STRUCTURAL-TIES PANEL
{
  "rows": 9612,
  "countries_or_historical_entities": 199,
  "years": [
    2000,
    2024
  ],
  "coverage_status_counts": {
    "bilateral_observed": 7977,
    "no_directed_record": 850,
    "unilateral_observed": 785
  },
  "direction_count_distribution": {
    "0": 850,
    "1": 785,
    "2": 7977
  },
  "baseline_score_summary": {
    "count": 9612.0,
    "mean": 0.838546543383271,
    "std": 0.31921989291733777,
    "min": 0.0,
    "25%": 0.84375,
    "50%": 1.0,
    "75%": 1.0,
    "max": 1.0
  },
  "validation_failures": {
    "duplicate_keys": 0,
    "blank_country_names": 0,
    "invalid_direction_counts": 0,
    "bilateral_flag_mismatches": 0,
    "diplomatic_score_mean_only_out_of_bounds": 0,
    "diplomatic_score_mean_reciprocity_75_25_out_of_bounds": 0,
    "diplomatic_score_mean_reciprocity_50_50_out_of_bounds": 0,
    "diplomatic_score_geometric_out_of_bounds": 0,
    "diplomatic_structural_score_raw_out_of_bounds": 0,
    "scaled_scor

In [3]:
# load production outputs
panel = pd.read_csv(PRODUCTION_PANEL, low_memory=False)
sensitivity = pd.read_csv(
    REPORT_DIRECTORY / "diplomatic_score_sensitivity_by_year.csv"
)
method_comparison = pd.read_csv(
    REPORT_DIRECTORY / "diplomatic_score_method_comparison.csv"
)

print("Production rows:", len(panel))
print("Countries or historical entities:", panel["country_iso3"].nunique())
print("Years:", int(panel["year"].min()), "to", int(panel["year"].max()))

Production rows: 9612
Countries or historical entities: 199
Years: 2000 to 2024


In [4]:
# validate keys, bounds, coverage and method identity
keys = ["country_iso3", "anchor", "year"]
score_columns = [
    "diplomatic_score_mean_only",
    "diplomatic_score_mean_reciprocity_75_25",
    "diplomatic_score_mean_reciprocity_50_50",
    "diplomatic_score_geometric",
    "diplomatic_structural_score_raw",
]

assert len(panel) == 9612
assert panel["country_iso3"].nunique() == 199
assert not panel.duplicated(keys).any()
assert panel["country_name"].notna().all()
assert panel["direction_count"].isin([0, 1, 2]).all()
assert (
    panel["bilateral_observed"]
    == panel["direction_count"].eq(2)
).all()

for column in score_columns:
    assert panel[column].between(0, 1).all()

assert panel["diplomatic_structural_score_scaled"].between(0, 100).all()
assert (
    panel["diplomatic_structural_score_raw"]
    == panel["diplomatic_score_mean_reciprocity_75_25"]
).all()

print("PASS: unique keys and complete canonical names")
print("PASS: direction counts and bilateral flags agree")
print("PASS: all raw scores lie in [0, 1]")
print("PASS: all scaled scores lie in [0, 100]")
print("PASS: the production score uses the 75-25 baseline")

PASS: unique keys and complete canonical names
PASS: direction counts and bilateral flags agree
PASS: all raw scores lie in [0, 1]
PASS: all scaled scores lie in [0, 100]
PASS: the production score uses the 75-25 baseline


In [5]:
# inspect coverage status
coverage = (
    panel["source_coverage_status"]
    .value_counts()
    .rename_axis("coverage_status")
    .reset_index(name="country_anchor_years")
)

direction_counts = (
    panel["direction_count"]
    .value_counts()
    .sort_index()
    .rename_axis("direction_count")
    .reset_index(name="country_anchor_years")
)

display(coverage)
display(direction_counts)

,coverage_status,country_anchor_years
0,bilateral_observed,7977
1,no_directed_record,850
2,unilateral_observed,785


,direction_count,country_anchor_years
0,0,850
1,1,785
2,2,7977


## Coverage interpretation

The production universe is inherited from the completed relationship-condition panel. Every country-anchor-year therefore remains available for final index integration.

Coverage states distinguish:

- `bilateral_observed`: both directed diplomatic records are present;
- `unilateral_observed`: exactly one directed record is present; and
- `no_directed_record`: neither directed record is present.

This distinction must remain available when scores are interpreted or thresholded.

In [6]:
# verify observed zero versus unobserved zero
observed_zero = panel[
    (
        panel["anchor_to_country_observed"]
        & panel["anchor_to_country_lor"].eq(0)
    )
    |
    (
        panel["country_to_anchor_observed"]
        & panel["country_to_anchor_lor"].eq(0)
    )
].copy()

unobserved_directions = int(
    (~panel["anchor_to_country_observed"]).sum()
    + (~panel["country_to_anchor_observed"]).sum()
)

print("Observed adverse zero-depth rows:", len(observed_zero))
print("Unobserved directed relationships:", unobserved_directions)

assert (
    observed_zero["anchor_to_country_adverse"]
    | observed_zero["country_to_anchor_adverse"]
).all()

print("PASS: observed adverse zero-depth records remain distinguishable")
display(
    observed_zero[
        [
            "country_iso3",
            "country_name",
            "anchor",
            "year",
            "anchor_to_country_observed",
            "anchor_to_country_lor",
            "country_to_anchor_observed",
            "country_to_anchor_lor",
            "bilateral_adverse",
        ]
    ].head(20)
)

Observed adverse zero-depth rows: 19
Unobserved directed relationships: 2485
PASS: observed adverse zero-depth records remain distinguishable


,country_iso3,country_name,anchor,year,anchor_to_country_observed,anchor_to_country_lor,country_to_anchor_observed,country_to_anchor_lor,bilateral_adverse
25,AFG,Afghanistan,United States,2000,True,0.0,False,0.000,False
46,AFG,Afghanistan,United States,2021,True,0.0,True,0.000,True
475,AUT,Austria,United States,2000,True,0.0,True,1.000,False
1046,BLR,Belarus,United States,2021,True,0.0,True,0.750,False
1047,BLR,Belarus,United States,2022,True,0.0,True,0.750,False
1391,CAF,Central African Republic,United States,2014,True,0.0,True,1.000,False
1392,CAF,Central African Republic,United States,2015,True,0.0,True,1.000,False
3927,IRN,Iran,United States,2015,True,0.0,True,0.125,False
3928,IRN,Iran,United States,2016,True,0.0,True,0.125,False
3930,IRN,Iran,United States,2018,False,0.0,True,0.000,False


In [7]:
# inspect baseline distribution by anchor and year
latest_year = int(panel["year"].max())
latest = panel[panel["year"].eq(latest_year)]

latest_summary = (
    latest.groupby("anchor")["diplomatic_structural_score_raw"]
    .describe()
)

annual_summary = sensitivity.tail(10)

print("Latest year:", latest_year)
display(latest_summary)
display(annual_summary)

Latest year: 2024


,count,mean,std,min,25%,50%,75%,max
anchor,,,,,,,,
China,196.0,0.866869,0.316368,0.0,1.00000,1.0,1.0,1.0
United States,196.0,0.839923,0.260026,0.0,0.84375,1.0,1.0,1.0


,year,anchor,observations,baseline_mean,mean_only_mean,equal_weight_mean,geometric_mean,bilateral_observed,unilateral_observed,no_directed_record
40,2020,China,196,0.863839,0.867347,0.860332,0.856868,169,4,23
41,2020,United States,196,0.825096,0.850128,0.800064,0.796027,168,20,8
42,2021,China,196,0.859375,0.863520,0.855230,0.851083,168,6,22
43,2021,United States,196,0.830676,0.853316,0.808036,0.798738,168,20,8
44,2022,China,196,0.855389,0.860332,0.850446,0.850033,169,5,22
45,2022,United States,196,0.850925,0.869260,0.832589,0.814142,166,21,9
46,2023,China,196,0.872927,0.876913,0.868941,0.868665,172,3,21
47,2023,United States,196,0.864796,0.881378,0.848214,0.831964,168,19,9
48,2024,China,196,0.866869,0.871811,0.861926,0.860920,171,5,20
49,2024,United States,196,0.839923,0.860969,0.818878,0.813233,169,18,9


## Distribution note

The diplomatic score is expected to be concentrated near the top of the scale because full ambassadorial, single-focus representation is common. This is a substantive feature of diplomatic representation, not a reason to manufacture additional variation.

The later threshold step must therefore examine ties, score mass points, and classification stability. Jenks natural breaks may be informative, but only after diplomatic, economic, technology, and composite distributions are available.

In [8]:
# compare scoring methods for asymmetric relations
asymmetric = method_comparison[
    method_comparison["depth_gap"].gt(0)
].copy()

asymmetric["absolute_baseline_discount_from_mean"] = (
    asymmetric["baseline_minus_mean_only"].abs()
)

most_sensitive = asymmetric.sort_values(
    "absolute_baseline_discount_from_mean",
    ascending=False,
).head(25)

print("Asymmetric country-anchor-years:", len(asymmetric))
print(
    "Maximum baseline discount from mean-only:",
    most_sensitive["absolute_baseline_discount_from_mean"].max(),
)

display(
    most_sensitive[
        [
            "country_iso3",
            "country_name",
            "anchor",
            "year",
            "anchor_to_country_lor",
            "country_to_anchor_lor",
            "mean_depth",
            "reciprocity_depth",
            "diplomatic_score_mean_only",
            "diplomatic_score_mean_reciprocity_75_25",
            "diplomatic_score_mean_reciprocity_50_50",
            "diplomatic_score_geometric",
        ]
    ]
)

Asymmetric country-anchor-years: 1682
Maximum baseline discount from mean-only: 0.125


,country_iso3,country_name,anchor,year,anchor_to_country_lor,country_to_anchor_lor,mean_depth,reciprocity_depth,diplomatic_score_mean_only,diplomatic_score_mean_reciprocity_75_25,diplomatic_score_mean_reciprocity_50_50,diplomatic_score_geometric
4935,LCA,Saint Lucia,United States,2003,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0
8311,TCD,Chad,China,2016,1.0,0.0,0.5,0.0,0.5,0.375,0.25,0.0
3254,GNB,Guinea-Bissau,United States,2002,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0
3255,GNB,Guinea-Bissau,United States,2003,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0
3259,GNB,Guinea-Bissau,United States,2007,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0
3264,GNB,Guinea-Bissau,United States,2012,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0
3265,GNB,Guinea-Bissau,United States,2013,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0
3266,GNB,Guinea-Bissau,United States,2014,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0
3267,GNB,Guinea-Bissau,United States,2015,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0
3302,GNQ,Equatorial Guinea,United States,2000,0.0,1.0,0.5,0.0,0.5,0.375,0.25,0.0


In [9]:
# validate continuity diagnostics without using them in the baseline
continuity_columns = [
    "anchor_to_country_five_year_presence_rate",
    "country_to_anchor_five_year_presence_rate",
    "bilateral_five_year_presence_rate",
]

for column in continuity_columns:
    assert panel[column].between(0, 1).all()

assert panel["anchor_to_country_presence_streak"].ge(0).all()
assert panel["country_to_anchor_presence_streak"].ge(0).all()
assert panel["bilateral_presence_streak"].ge(0).all()

print("PASS: continuity diagnostics have valid bounds")
print("PASS: continuity does not enter the baseline formula")

PASS: continuity diagnostics have valid bounds
PASS: continuity does not enter the baseline formula


In [10]:
# compare China and United States within country-year
comparison = panel.pivot(
    index=["country_iso3", "country_name", "year"],
    columns="anchor",
    values="diplomatic_structural_score_raw",
).reset_index()

comparison["china_minus_us_diplomatic_score"] = (
    comparison["China"] - comparison["United States"]
)

latest_comparison = comparison[
    comparison["year"].eq(latest_year)
].copy()

largest_china_advantage = latest_comparison.nlargest(
    15,
    "china_minus_us_diplomatic_score",
)
largest_us_advantage = latest_comparison.nsmallest(
    15,
    "china_minus_us_diplomatic_score",
)

print("Largest China diplomatic-score advantages in the latest year")
display(largest_china_advantage)

print("Largest United States diplomatic-score advantages in the latest year")
display(largest_us_advantage)

Largest China diplomatic-score advantages in the latest year


anchor,country_iso3,country_name,year,China,United States,china_minus_us_diplomatic_score
24,AFG,Afghanistan,2024,1.00,0.00000,1.00000
1650,GNB,Guinea-Bissau,2024,1.00,0.00000,1.00000
1980,IRN,Iran,2024,1.00,0.00000,1.00000
3523,PRK,North Korea,2024,1.00,0.00000,1.00000
3984,STP,Sao Tome and Principe,2024,1.00,0.00000,1.00000
4159,SYR,Syria,2024,1.00,0.00000,1.00000
4655,VEN,Venezuela,2024,1.00,0.28125,0.71875
950,COM,Comoros,2024,1.00,0.37500,0.62500
1150,DMA,Dominica,2024,1.00,0.37500,0.62500
1725,GRD,Grenada,2024,1.00,0.37500,0.62500


Largest United States diplomatic-score advantages in the latest year


anchor,country_iso3,country_name,year,China,United States,china_minus_us_diplomatic_score
1750,GTM,Guatemala,2024,0.0,1.00000,-1.00000
2340,KSV,Kosovo,2024,0.0,1.00000,-1.00000
2587,LTU,Lithuania,2024,0.0,1.00000,-1.00000
2800,MHL,Marshall Islands,2024,0.0,1.00000,-1.00000
3573,PRY,Paraguay,2024,0.0,1.00000,-1.00000
549,BLZ,Belize,2024,0.0,0.84375,-0.84375
1855,HTI,Haiti,2024,0.0,0.84375,-0.84375
3448,PLW,Palau,2024,0.0,0.84375,-0.84375
4109,SWZ,Swaziland,2024,0.0,0.84375,-0.84375
4605,VAT,Holy See,2024,0.0,0.84375,-0.84375


## Production conclusion

The diplomatic structural-ties component now provides:

- a complete country-anchor-year panel aligned with the relationship-condition universe;
- two directional LOR values;
- explicit observed and adverse flags;
- mean depth, reciprocity depth, and asymmetry;
- the 75-25 baseline score;
- three sensitivity scores;
- continuity diagnostics; and
- direct China-versus-US comparability.

The score remains continuous. No low, medium, or high thresholds are assigned at this stage.

# Decision register

## Decided

- Use source-provided LOR rather than reconstructing a second representation scale.
- Score missing directed records as zero operational depth while retaining an observed flag.
- Keep observed LOR zero distinct from unobserved direction zero.
- Use 75% mean depth and 25% reciprocity depth as the baseline.
- Retain mean-only, equal-component-weight, and geometric sensitivity scores.
- Exclude continuity from the baseline while retaining diagnostics.
- Keep relationship condition separate from diplomatic structural depth.
- Delay thresholds until all component and composite scores are available.

## Next

- Review score distributions and country-level edge cases.
- Lock the diplomatic production panel.
- Move to economic structural ties.
- Build the technology component.
- Integrate component scores.
- Compare threshold methods, including Jenks natural breaks.